# JerseyNet ablation study

Which parts of a modern detector recipe help JerseyNet? Part of
[MHKhan50/jersey-number-detection](https://github.com/MHKhan50/jersey-number-detection).

1. **Stage 1:** six variants, seed 0 each: baseline, mosaic augmentation, GIoU box loss,
   384 px input, mosaic + GIoU, and mosaic + GIoU + 384 px.
2. **Stage 2:** the variant with the best **validation** AP50:95 is trained with seeds 1 and 2.
   The test split is never used for the choice.

Every run is scored with the same evaluator as the main results (COCO-style AP, first-box IoU,
95% bootstrap intervals).

**Kaggle settings:** Accelerator *GPU T4 x2*, Internet *on*, the `jersey-2` dataset attached under
*Input*. Then *Save Version → Save & Run All (Commit)*. Expected run time is about 4 to 5 hours;
results go to `/kaggle/working/ablation`.

In [ ]:
# Settings
import os

REPO_URL = "https://github.com/MHKhan50/jersey-number-detection.git"
REPO_REF = "main"                    # branch or tag to train from
CONFIG = "configs/jersey_net.json"   # baseline recipe; variants change one setting each
TIME_BUDGET_HOURS = 10.5             # no new run starts after this; Kaggle stops at 12 h
OUT_DIR = "/kaggle/working/ablation"
INPUT_ROOTS = ["/kaggle/input"]
DOWNLOAD_DIR = "/tmp/jersey-2"

# Continuous-integration smoke test only: a tiny synthetic dataset and two epochs.
SMOKE = os.environ.get("JND_SMOKE") == "1"
if SMOKE:
    TIME_BUDGET_HOURS = 0.5
    OUT_DIR = os.environ.get("JND_OUT", "/tmp/jnd-smoke/ablation")
print({"REPO_REF": REPO_REF, "CONFIG": CONFIG, "TIME_BUDGET_HOURS": TIME_BUDGET_HOURS, "SMOKE": SMOKE})

In [ ]:
# Environment check
import subprocess

import torch

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i))
    major, minor = torch.cuda.get_device_capability(0)
    if f"sm_{major}{minor}" not in torch.cuda.get_arch_list():
        print("WARNING: this PyTorch build has no kernels for this GPU. "
              "Switch the accelerator to GPU T4 x2. Training would fall back to the CPU.")
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("WARNING: no GPU. Set Accelerator to GPU T4 x2 in the notebook settings.")

In [ ]:
# Get the code (the commit is recorded in ablation/summary.json)
import sys
import time
from pathlib import Path

REPO_DIR = Path(os.environ.get("JND_REPO_DIR", "/tmp/jersey-number-detection"))
if not (REPO_DIR / "src" / "jersey_detect").exists():
    for attempt in range(3):
        r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)])
        if r.returncode == 0:
            break
        time.sleep(10)
SRC = str(REPO_DIR / "src")
sys.path.insert(0, SRC)
import jersey_detect

print("jersey_detect", jersey_detect.__version__, "from", SRC)
print(subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--format=%H %s"],
                     capture_output=True, text=True).stdout)

In [ ]:
# Find the dataset (Kaggle input first, then Roboflow with the ROBOFLOW_API_KEY secret)
from jersey_detect.prepare import locate_or_download

DATA_ROOT = None
try:
    if SMOKE:
        from jersey_detect.synthetic import make_dataset
        DATA_ROOT = make_dataset("/tmp/jnd-smoke/data")
    else:
        DATA_ROOT = locate_or_download(INPUT_ROOTS, DOWNLOAD_DIR)
    print("Dataset:", DATA_ROOT)
    for split in ("train", "valid", "val", "test"):
        p = Path(DATA_ROOT) / split / "images"
        if p.is_dir():
            print(f"  {split}: {sum(1 for _ in p.iterdir())} images")
except Exception as exc:
    print("DATASET NOT FOUND:", exc)
    # Stop here so the version is marked as failed instead of finishing without training.
    raise RuntimeError("No dataset: attach jersey-2 as an input") from exc

In [ ]:
# Run the ablation in a separate process, so a crash cannot take the notebook down.
# Output streams here and is also saved to ablation/run.log.
return_code = None
if DATA_ROOT is not None:
    env = dict(os.environ, PYTHONPATH=SRC, PYTHONUNBUFFERED="1")
    cmd = [sys.executable, "-m", "jersey_detect", "ablate", "--data", str(DATA_ROOT),
           "--out", OUT_DIR, "--config", str(REPO_DIR / CONFIG), "--hours", str(TIME_BUDGET_HOURS)]
    if SMOKE:
        cmd.append("--smoke")
    print(" ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="", flush=True)
    return_code = proc.wait()
print("return code:", return_code)

In [ ]:
# Results
import json

summary_path = Path(OUT_DIR) / "summary.json"
if summary_path.exists():
    print((Path(OUT_DIR) / "results.md").read_text())
    summary = json.loads(summary_path.read_text())
    if summary["errors"]:
        print("Errors:", json.dumps(summary["errors"], indent=1))
    for p in sorted(Path(OUT_DIR).rglob("*")):
        if p.is_file():
            print(f"{p.stat().st_size / 1e6:8.2f} MB  {p.relative_to(OUT_DIR)}")
else:
    print("No summary written; see the output of the previous cells.")